In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns 
import plotly.express as px

In [2]:
df = pd.read_excel("Raw_Online_Retail.xlsx")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [4]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


### Data Cleaning

In [5]:
# drop missing values across all column
df.dropna(inplace=True) 

# keep only positive quantity
df = df[df["Quantity"] > 0]

# Keep only positive unit prices
df = df[df['UnitPrice'] > 0]

# Standardize InvoiceDate Column
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 397884 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    397884 non-null  object        
 1   StockCode    397884 non-null  object        
 2   Description  397884 non-null  object        
 3   Quantity     397884 non-null  int64         
 4   InvoiceDate  397884 non-null  datetime64[ns]
 5   UnitPrice    397884 non-null  float64       
 6   CustomerID   397884 non-null  float64       
 7   Country      397884 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 27.3+ MB


In [7]:
df.isna().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

In [8]:
df.to_csv("cleaned_online_retail.csv", index=False)

### Feature Extraction

In [9]:
#cleaned_df = pd.read_csv("cleaned_online_retail.csv")

In [10]:
# Creating the revenue feature

df["Revenue"] = df['Quantity'] * df['UnitPrice']

In [11]:
## Extract data-time columns 
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["MonthName"] = df['InvoiceDate'].dt.month_name()
df["DayName"] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df["InvoiceDate"].dt.hour

#converting customerID to string
df["CustomerID"] = df["CustomerID"].astype(str)

In [12]:
#value counts
df['Year'].value_counts()

Year
2011    371727
2010     26157
Name: count, dtype: int64

In [13]:
# For monthly trend analysis
df['YearMonth'] = df['InvoiceDate'].dt.to_period("M").astype(str)

### Initial Data Exploration

In [14]:
#number of rows
len(df)

# Number of Transactions
print(df['InvoiceNo'].nunique())

# Number of products
print(df['StockCode'].nunique())

# Number of Customers 
print(df['CustomerID'].nunique())

18532
3665
4338


### Create Core Buisiness KPIs

In [15]:
total_revenue = df["Revenue"].sum()

total_orders = df['InvoiceNo'].nunique()

total_customers = df['CustomerID'].nunique()

total_products = df['StockCode'].nunique()

total_quantity = df['Quantity'].sum()

print("Revenue: ", total_revenue)
print("Orders: ", total_orders)
print("Customers: ", total_customers)
print("Products: ", total_products)
print("Quantity: ", total_quantity)

Revenue:  8911407.904
Orders:  18532
Customers:  4338
Products:  3665
Quantity:  5167812


### EDA 

#### Average Order Value

In [16]:
# One invoice represents an order 
order_revenue = df.groupby("InvoiceNo")['Revenue'].sum()

#Average order value
average_order_value = order_revenue.mean()
average_order_value

np.float64(480.8659563997409)

#### Average Items per Order

In [17]:
average_item_per_order = df.groupby("InvoiceNo")['Quantity'].sum().mean()
average_item_per_order

np.float64(278.8588387653788)

#### Customer Classification

In [18]:
customer_revenue = (df.groupby("CustomerID")['Revenue']
                    .sum().reset_index())
customer_revenue

,CustomerID,Revenue
0,12346.0,77183.60
1,12347.0,4310.00
2,12348.0,1797.24
3,12349.0,1757.55
4,12350.0,334.40
...,...,...
4333,18280.0,180.60
4334,18281.0,80.82
4335,18282.0,178.05
4336,18283.0,2094.88


In [19]:
def classify_customer(revenue):
    if revenue >= 10000:
        return "High Value"
    elif revenue >= 5000:
        return "Medium Value"
    else:
        return "Low Value"

In [20]:
customer_revenue["CustomerClass"] = (
    customer_revenue["Revenue"].apply(classify_customer)
    )

In [21]:
customer_revenue.head(2)

,CustomerID,Revenue,CustomerClass
0,12346.0,77183.6,High Value
1,12347.0,4310.0,Low Value


In [22]:
# Merging CustomerClass with df

df = df.merge(customer_revenue[['CustomerID','CustomerClass']], 
              on="CustomerID", how="left")

df.head(2)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,Year,Month,MonthName,DayName,Hour,YearMonth,CustomerClass
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,2010,12,December,Wednesday,8,2010-12,Medium Value
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,2010,12,December,Wednesday,8,2010-12,Medium Value


### How is Revenue changing over time?

In [ ]:
monthly_revenue = (
    df.groupby("YearMonth")['Revenue'].sum().reset_index()
)

fig = px.line(monthly_revenue, x="YearMonth", y="Revenue", 
              markers=True, title="Monthly Revenue Trend")
fig.show()

#### Which Country Generate the most revenue?

In [24]:
country_sales = (
    df.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

country_sales

,Country,Revenue
0,United Kingdom,7308391.554
1,Netherlands,285446.340
2,EIRE,265545.900
3,Germany,228867.140
4,France,209024.050
5,Australia,138521.310
6,Spain,61577.110
7,Switzerland,56443.950
8,Belgium,41196.340
9,Sweden,38378.330


In [25]:
#Visualize 
fig = px.bar(
    country_sales, 
    x="Revenue",
    y ="Country", 
    orientation="h",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

#### Which Products Generate the Most Revenue

In [26]:
product_revenue = (
    df.groupby("Description")
    .agg(
        Revenue =('Revenue', "sum"),
        Quantity = ("Quantity", "sum")
    )
    .sort_values("Revenue", ascending=False)
    .reset_index()
)

product_revenue.head(10)

,Description,Revenue,Quantity
0,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995
1,REGENCY CAKESTAND 3 TIER,142592.95,12402
2,WHITE HANGING HEART T-LIGHT HOLDER,100448.15,36725
3,JUMBO BAG RED RETROSPOT,85220.78,46181
4,MEDIUM CERAMIC TOP STORAGE JAR,81416.73,77916
5,POSTAGE,77803.96,3120
6,PARTY BUNTING,68844.33,15291
7,ASSORTED COLOUR BIRD ORNAMENT,56580.34,35362
8,Manual,53779.93,7173
9,RABBIT NIGHT LIGHT,51346.20,27202


In [27]:
top_products = product_revenue.head(5)

fig = px.bar(
    top_products,
    x="Revenue",
    y="Description",
    orientation="h",
    title="Top 5 Products by Revenue"
)

fig.update_layout(yaxis={"categoryorder" : "total ascending"})
fig.show()

#### Revenue Percentage by Customer Class

In [28]:
class_summary = (customer_revenue.groupby("CustomerClass")
.agg(
    NumberOfCustomers = ("CustomerID", "nunique"), 
    TotalRevenue = ("Revenue", "sum")
)
.reset_index()
)

In [29]:
class_summary

,CustomerClass,NumberOfCustomers,TotalRevenue
0,High Value,104,3653544.480
1,Low Value,4063,4109636.384
2,Medium Value,171,1148227.040


In [30]:
## % Revenue by CustomerClass
class_summary["RevenuePercentage"] = (
    class_summary['TotalRevenue'] / 
    class_summary['TotalRevenue'].sum() * 100
)

class_summary

,CustomerClass,NumberOfCustomers,TotalRevenue,RevenuePercentage
0,High Value,104,3653544.480,40.998510
1,Low Value,4063,4109636.384,46.116578
2,Medium Value,171,1148227.040,12.884912


In [ ]:
fig = px.pie(
    class_summary, 
    names = "CustomerClass",
    values="TotalRevenue",
    title="Revenue Share by Customer Class",
    hole=0.4
)

fig.show()

In [49]:
customer_data = df.groupby("CustomerID").agg(
                Revenue = ("Revenue", "sum"),
                Orders=("InvoiceNo", "nunique"),
                Units=("Quantity", "sum")         
                    )
customer_data.reset_index()           

,CustomerID,Revenue,Orders,Units
0,12346.0,77183.60,1,74215
1,12347.0,4310.00,7,2458
2,12348.0,1797.24,4,2341
3,12349.0,1757.55,1,631
4,12350.0,334.40,1,197
...,...,...,...,...
4333,18280.0,180.60,1,45
4334,18281.0,80.82,1,54
4335,18282.0,178.05,2,103
4336,18283.0,2094.88,16,1397


### Which Days Generate the Most Revenue?

In [39]:
day_revenue = (
    df.groupby("DayName")["Revenue"].sum()
    .reset_index()
)

day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

## Reorder DayName using the day_order
day_revenue['DayName'] = pd.Categorical(
    day_revenue['DayName'],
    categories=day_order,
    ordered=True
)

day_revenue = day_revenue.sort_values("DayName")

fig = px.bar(
    day_revenue,
    x="DayName",
    y="Revenue",
    title="Revenue by Day of the week"
)

fig.show()

### What Time Do Customers Shop?

In [45]:

## We need to get how many orders in each hour using Unique InvoiceNo
hourly_orders = (
    df.groupby("Hour")['InvoiceNo'].nunique()
    .reset_index(name="Orders")
)

hourly_orders

,Hour,Orders
0,6,1
1,7,29
2,8,555
3,9,1393
4,10,2226
5,11,2277
6,12,3130
7,13,2636
8,14,2274
9,15,2037


In [47]:
fig = px.line(
    hourly_orders,
    x="Hour",
    y="Orders",
    markers=True,
    title = "Orders by Hour of Day"
)

fig.show()